# Can we predict late delivery?
Target: `late_delivery`. Intended use: predict at dispatch time for future consignments, so the split is **temporal**.

Decisions taken from `columns.csv` / `checklist.md`:
- `consignment` is an identifier: dropped. Every id is unique, so there are no repeated entities and a time split is the relevant structure.
- `claim_paid_eur` is recorded after delivery (a claim is settled after the fact), so it is **not known at prediction time** and is excluded. It leaks the target.
- Features: `carrier`, `distance_km`, `weight_kg`, `parcels`, plus hour and weekday derived from `dispatched_on`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss

df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
df['hour'] = df.dispatched_on.dt.hour
df['weekday'] = df.dispatched_on.dt.weekday
print(df.shape, df.dispatched_on.min(), df.dispatched_on.max(), df.consignment.is_unique)
print(df.late_delivery.mean())

## 1. Test set: the latest 20% of rows (set aside before any modelling)

In [ ]:
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(len(train), len(test), train.dispatched_on.max(), test.dispatched_on.min())
FEATS = ['carrier', 'distance_km', 'weight_kg', 'parcels', 'hour', 'weekday']
CAT, NUM = ['carrier'], ['distance_km', 'weight_kg', 'parcels', 'hour', 'weekday']
Xtr, ytr = train[FEATS], train.late_delivery.values
Xte, yte = test[FEATS], test.late_delivery.values
print('train late rate', ytr.mean())

## 2. Candidates (named before any scoring)
1. `baseline`: predicts the training late rate for every row.
2. `logreg_C0.1`, `logreg_C1`, `logreg_C10`: logistic regression (scaled numerics, one-hot carrier).
3. `hgb_small`: gradient boosting, depth 3, lr 0.05, 150 iters.
4. `hgb_large`: gradient boosting, depth 6, lr 0.05, 300 iters.

Selection metric: mean ROC AUC over expanding-window time-series CV folds within the training rows. Preprocessing is inside a `Pipeline`, so it is refit within each fold. The decision threshold is chosen afterwards on pooled out-of-fold predictions (maximising F1).

In [ ]:
def prep():
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), CAT),
                              ('n', StandardScaler(), NUM)])

def make(name):
    if name == 'baseline':
        return Pipeline([('m', DummyClassifier(strategy='prior'))])
    if name.startswith('logreg'):
        return Pipeline([('p', prep()), ('m', LogisticRegression(C=float(name.split('C')[1]), max_iter=1000))])
    if name == 'hgb_small':
        return Pipeline([('p', prep()), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=0))])
    if name == 'hgb_large':
        return Pipeline([('p', prep()), ('m', HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=300, random_state=0))])

CANDS = ['baseline', 'logreg_C0.1', 'logreg_C1', 'logreg_C10', 'hgb_small', 'hgb_large']
tscv = TimeSeriesSplit(n_splits=5)
oof, rows = {}, []
for n in CANDS:
    p = np.full(len(train), np.nan)
    aucs = []
    for a, b in tscv.split(Xtr):
        m = make(n).fit(Xtr.iloc[a], ytr[a])
        p[b] = m.predict_proba(Xtr.iloc[b])[:, 1]
        aucs.append(roc_auc_score(ytr[b], p[b]))
    oof[n] = p
    rows.append((n, np.mean(aucs), np.std(aucs)))
cv = pd.DataFrame(rows, columns=['model', 'cv_auc_mean', 'cv_auc_sd']).sort_values('cv_auc_mean', ascending=False)
cv

## 3. Choose model and threshold on CV only

In [ ]:
best = cv.iloc[0].model
mask = ~np.isnan(oof[best])
grid = np.linspace(0.05, 0.8, 76)
f1s = [f1_score(ytr[mask], oof[best][mask] >= t) for t in grid]
thr = float(grid[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr, 3), '| CV F1 at threshold:', round(max(f1s), 3))
print('CV F1 of always-late:', round(f1_score(ytr[mask], np.ones(mask.sum())), 3))

**Recorded choice (made from the CV results above, before touching the test set):** the model in `best`, features `carrier, distance_km, weight_kg, parcels, hour, weekday`, and the threshold `thr`.

## 4. Refit on all training rows, score the test set once

In [ ]:
final = make(best).fit(Xtr, ytr)
pt = final.predict_proba(Xte)[:, 1]
base = make('baseline').fit(Xtr, ytr).predict_proba(Xte)[:, 1]
res = pd.DataFrame({'ROC AUC': [roc_auc_score(yte, pt), roc_auc_score(yte, base)],
                    'PR AUC': [average_precision_score(yte, pt), average_precision_score(yte, base)],
                    'Brier': [brier_score_loss(yte, pt), brier_score_loss(yte, base)],
                    'F1 @ chosen thr': [f1_score(yte, pt >= thr), f1_score(yte, base >= thr)]},
                   index=[best + ' (headline)', 'baseline'])
print('test late rate', yte.mean())
res.round(3)

## Conclusion
The headline is the first row of the table above: test ROC AUC / PR AUC versus the trivial baseline (AUC 0.5, PR AUC about the late rate). The test set was scored once, with model, features and threshold fixed from cross-validation. **Caveat (observed after scoring, no model change made):** the late rate is not stable over time. Monthly late rates run 10-49% through Sep 2025, then fall to about 1-5% from Oct 2025, so the test period (late rate about 2%) has a very different base rate from training (about 26%). Ranking ability (ROC AUC) is what carries over; the chosen threshold and calibration do not, and F1/Brier at that threshold are weak. Predictions need recalibration on recent data before operational use. `claim_paid_eur` was deliberately excluded: it would look near-perfect but is only known after the outcome.